# Data Cleaning Project — Messy Customer Sales Dataset
**Track:** Data Analytics (Level 1, Task 3)

**Objective:** Take a deliberately messy customer sales dataset and systematically transform it into a clean, analysis-ready dataset, documenting every decision made along the way.

**Dataset:** `messy_customer_sales.csv` — a synthetic customer/sales dataset containing the kinds of real-world issues analysts routinely face: missing values, duplicate rows, inconsistent categorical formatting, mixed date formats, incorrect data types, and outliers.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
df = pd.read_csv('messy_customer_sales.csv')
print('Shape:', df.shape)
df.head(10)

Shape: (625, 9)


   CustomerID            Name   Age  Gender       City     JoinDate  \
0        1001      Customer_0  20.0  female     Mumbai   01-01-2022   
1        1002      Customer_1  57.0    Male  bengaluru   02/01/2022   
2        1003      Customer_2  51.0  Female  Bengaluru  04-Jan-2022   
3        1004      Customer_3  39.0  Female      Delhi   05/01/2022   
4        1005      Customer_4  39.0       F    mumbai    01-07-2022   
5        1006      Customer_5  62.0  Female     Mumbai  08-Jan-2022   
6        1007      Customer_6  20.0       M     Mumbai   01-10-2022   
7        1008    Customer_7     NaN  Female  bengaluru   01-11-2022   
8        1009      Customer_8  26.0    Male    mumbai   13-Jan-2022   
9        1010      Customer_9  21.0    Male    Kolkata  14-Jan-2022   

  PurchaseAmount ProductCategory  Rating  
0      $1,613.36      home decor     4.0  
1        1725.24       clothing      5.0  
2        4711.68     Electronics     3.0  
3        4121.57       clothing      2.0  
4  

## 1. Data Quality Report
Before touching anything, let's get a full picture of what's wrong: nulls per column, duplicate rows, dtype issues, and obvious value-range anomalies.

In [2]:
print('--- Column dtypes ---')
print(df.dtypes)
print()
print('--- Null counts per column ---')
print(df.isnull().sum())
print()
print('--- Duplicate rows ---')
print('Full duplicates:', df.duplicated().sum())
print()
print('--- Unique values in categorical columns (raw) ---')
for col in ['Gender', 'City', 'ProductCategory']:
    print(f"{col}: {sorted(df[col].dropna().unique().tolist())}")


--- Column dtypes ---
CustomerID           int64
Name                   str
Age                float64
Gender                 str
City                   str
JoinDate               str
PurchaseAmount         str
ProductCategory        str
Rating             float64
dtype: object

--- Null counts per column ---
CustomerID          0
Name                0
Age                36
Gender             59
City               36
JoinDate            0
PurchaseAmount     38
ProductCategory     0
Rating             39
dtype: int64

--- Duplicate rows ---
Full duplicates: 24

--- Unique values in categorical columns (raw) ---
Gender: ['F', 'Female', 'M', 'Male', 'female', 'male']
City: [' Delhi', 'Bengaluru', 'Chennai', 'Chennai ', 'Delhi', 'Hyderabad', 'Kolkata', 'MUMBAI', 'Mumbai', 'Pune', 'bengaluru', 'hyderabad', 'mumbai ']
ProductCategory: ['Books', 'Clothing', 'Electronics', 'Groceries', 'Home Decor', 'clothing ', 'electronics', 'home decor']


In [3]:
print('--- Sample of PurchaseAmount raw values (dtype issue: mixed strings/numbers) ---')
print(df['PurchaseAmount'].apply(type).value_counts())
print(df['PurchaseAmount'].dropna().sample(10, random_state=1).tolist())
print()
print('--- Sample of JoinDate raw values (mixed date formats) ---')
print(df['JoinDate'].sample(10, random_state=1).tolist())


--- Sample of PurchaseAmount raw values (dtype issue: mixed strings/numbers) ---
PurchaseAmount
<class 'str'>      587
<class 'float'>     38
Name: count, dtype: int64
['3376.44', '1342.25', '2828.83', '1084.72', '$1,802.81', '2701.57', '$1,613.36', '2401.77', '1458.28', '1871.19']

--- Sample of JoinDate raw values (mixed date formats) ---
['2024-04-19', '2022-02-04', '03-04-2022', '06-28-2024', '17-Jul-2023', '09-Apr-2022', '10/09/2023', '05-14-2022', '06-23-2024', '19-Sep-2022']


**Data quality report — summary of what we found:**

- `Age`, `Rating`, `PurchaseAmount`, `City` all contain missing values (~6% each, injected).
- There are duplicate rows in the dataset (exact row-level duplicates).
- `Gender` has 7 different representations of just two categories (`Male`, `male`, `M`, `Female`, `female`, `F`, missing).
- `City` and `ProductCategory` have inconsistent casing and stray whitespace (e.g. `'mumbai '`, `'MUMBAI'`, `'Delhi'` vs `' Delhi'`).
- `PurchaseAmount` is stored as an `object` column because some values are formatted as currency strings (e.g. `'$1,613.36'`) while others are plain floats — this must be numeric for any analysis.
- `JoinDate` mixes at least four different date string formats (`YYYY-MM-DD`, `DD/MM/YYYY`, `MM-DD-YYYY`, `DD-Mon-YYYY`).
- `Name` has leading/trailing whitespace in some entries.
- `PurchaseAmount` and `Age` both contain implausible values (negative purchase amounts, purchase amounts in the tens of thousands, ages over 100 or negative) — likely outliers or data-entry errors.

## 2. Missing Data Handling
Strategy is chosen **per column**, based on what the column represents and how it will be used downstream.

| Column | Strategy | Justification |
|---|---|---|
| `Age` | Median imputation | Age is numeric and roughly symmetric; median is robust to the outliers we already spotted, so we don't let extreme ages skew the fill value. |
| `Rating` | Mode imputation | Rating is a discrete 1–5 scale — the mode (most common rating) is a more meaningful fill than a fractional mean. |
| `PurchaseAmount` | Median imputation (after cleaning to numeric) | Purchase amounts are right-skewed with outliers, so median (not mean) avoids inflating the fill value. |
| `City` | Row-level flag `'Unknown'` (not dropped) | City is categorical with no natural 'typical' value: imputing a specific city would fabricate information. Keeping an explicit `'Unknown'` category preserves the row for analyses that don't depend on location. |


In [4]:
df_clean = df.copy()

# Age -> median
age_median = df_clean['Age'].median()
df_clean['Age'] = df_clean['Age'].fillna(age_median)

# Rating -> mode
rating_mode = df_clean['Rating'].mode()[0]
df_clean['Rating'] = df_clean['Rating'].fillna(rating_mode)

# City -> explicit 'Unknown' category (filled after standardisation below)
df_clean['City'] = df_clean['City'].fillna('Unknown')

print(f'Age median used for imputation: {age_median}')
print(f'Rating mode used for imputation: {rating_mode}')
print('Remaining nulls (PurchaseAmount handled after dtype fix):')
print(df_clean.isnull().sum())

Age median used for imputation: 42.0
Rating mode used for imputation: 2.0
Remaining nulls (PurchaseAmount handled after dtype fix):
CustomerID          0
Name                0
Age                 0
Gender             59
City                0
JoinDate            0
PurchaseAmount     38
ProductCategory     0
Rating              0
dtype: int64


## 3. Fix Data Types — `PurchaseAmount`
`PurchaseAmount` must become numeric before we can impute or analyse it. We strip `$` and `,` characters, then convert to float.

In [5]:
def clean_currency(val):
    if pd.isna(val):
        return np.nan
    if isinstance(val, (int, float)):
        return float(val)
    return float(str(val).replace('$', '').replace(',', '').strip())

df_clean['PurchaseAmount'] = df_clean['PurchaseAmount'].apply(clean_currency)

purchase_median = df_clean['PurchaseAmount'].median()
df_clean['PurchaseAmount'] = df_clean['PurchaseAmount'].fillna(purchase_median)

print(f'PurchaseAmount median used for imputation: {purchase_median:.2f}')
print(df_clean['PurchaseAmount'].dtype)
print('Nulls remaining across dataset:', df_clean.isnull().sum().sum())

PurchaseAmount median used for imputation: 2461.68
float64
Nulls remaining across dataset: 59


## 4. Duplicate Removal

In [6]:
dupes_before = df_clean.duplicated().sum()
rows_before = len(df_clean)
df_clean = df_clean.drop_duplicates()
rows_after = len(df_clean)

print(f'Duplicate rows found: {dupes_before}')
print(f'Rows before: {rows_before} -> Rows after: {rows_after}')
print(f'Rows removed: {rows_before - rows_after}')

Duplicate rows found: 24
Rows before: 625 -> Rows after: 601
Rows removed: 24


## 5. Standardise Inconsistent Formatting
- `Name`: strip leading/trailing whitespace.
- `Gender`: collapse `Male`/`male`/`M` → `Male` and `Female`/`female`/`F` → `Female`.
- `City`: strip whitespace, title-case for consistent capitalisation.
- `ProductCategory`: strip whitespace, title-case.
- `JoinDate`: parse all mixed formats into a single `datetime64` dtype.

In [7]:
# Name
df_clean['Name'] = df_clean['Name'].str.strip()

# Gender
gender_map = {'male': 'Male', 'Male': 'Male', 'M': 'Male',
              'female': 'Female', 'Female': 'Female', 'F': 'Female'}
df_clean['Gender'] = df_clean['Gender'].map(gender_map)
df_clean['Gender'] = df_clean['Gender'].fillna('Unknown')

# City / ProductCategory
df_clean['City'] = df_clean['City'].str.strip().str.title()
df_clean['ProductCategory'] = df_clean['ProductCategory'].str.strip().str.title()

print('Gender values after standardisation:', df_clean['Gender'].unique().tolist())
print('City values after standardisation:', sorted(df_clean['City'].unique().tolist()))
print('ProductCategory values after standardisation:', sorted(df_clean['ProductCategory'].unique().tolist()))

Gender values after standardisation: ['Female', 'Male', 'Unknown']
City values after standardisation: ['Bengaluru', 'Chennai', 'Delhi', 'Hyderabad', 'Kolkata', 'Mumbai', 'Pune', 'Unknown']
ProductCategory values after standardisation: ['Books', 'Clothing', 'Electronics', 'Groceries', 'Home Decor']


In [8]:
# JoinDate: parse mixed formats robustly
df_clean['JoinDate'] = pd.to_datetime(df_clean['JoinDate'], format='mixed', dayfirst=False, errors='coerce')
print(df_clean['JoinDate'].dtype)
print('Unparseable dates (NaT) after conversion:', df_clean['JoinDate'].isna().sum())
df_clean[['JoinDate']].head()

datetime64[us]
Unparseable dates (NaT) after conversion: 0


    JoinDate
0 2022-01-01
1 2022-02-01
2 2022-01-04
3 2022-05-01
4 2022-01-07

## 6. Outlier Detection (IQR Method)
We use the IQR method on `Age` and `PurchaseAmount` to flag implausible values. 

**Decision:**
- `Age`: values below 0 or above 100 are impossible — these are **capped** to the nearest realistic bound (0–100) rather than dropped, since the rest of the row's data is still usable.
- `PurchaseAmount`: extreme values (negative amounts, or amounts far beyond the IQR fence) are **capped at the IQR fences** rather than removed, to avoid losing rows while still preventing them from distorting downstream statistics.

In [9]:
def iqr_bounds(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

# Age: apply logical bounds (0-100) — a real-world sanity check
age_outliers = df_clean[(df_clean['Age'] < 0) | (df_clean['Age'] > 100)]
print(f'Age outliers found (outside 0-100): {len(age_outliers)}')
df_clean['Age'] = df_clean['Age'].clip(lower=0, upper=100)

# PurchaseAmount: IQR-based capping
low, high = iqr_bounds(df_clean['PurchaseAmount'])
purchase_outliers = df_clean[(df_clean['PurchaseAmount'] < low) | (df_clean['PurchaseAmount'] > high)]
print(f'PurchaseAmount IQR bounds: [{low:.2f}, {high:.2f}]')
print(f'PurchaseAmount outliers found: {len(purchase_outliers)}')
df_clean['PurchaseAmount'] = df_clean['PurchaseAmount'].clip(lower=max(0, low), upper=high)

print('Age range after capping:', df_clean['Age'].min(), '-', df_clean['Age'].max())
print('PurchaseAmount range after capping:', round(df_clean['PurchaseAmount'].min(),2), '-', round(df_clean['PurchaseAmount'].max(),2))

Age outliers found (outside 0-100): 5
PurchaseAmount IQR bounds: [121.30, 4748.78]
PurchaseAmount outliers found: 15
Age range after capping: 0.0 - 100.0
PurchaseAmount range after capping: 121.3 - 4748.78


## 7. Final Data Type Corrections

In [10]:
df_clean['CustomerID'] = df_clean['CustomerID'].astype(int)
df_clean['Age'] = df_clean['Age'].astype(int)
df_clean['Rating'] = df_clean['Rating'].astype(int)
df_clean['Gender'] = df_clean['Gender'].astype('category')
df_clean['City'] = df_clean['City'].astype('category')
df_clean['ProductCategory'] = df_clean['ProductCategory'].astype('category')

df_clean.dtypes

CustomerID                  int64
Name                          str
Age                         int64
Gender                   category
City                     category
JoinDate           datetime64[us]
PurchaseAmount            float64
ProductCategory          category
Rating                      int64
dtype: object

## 8. Before vs. After Summary

In [11]:
summary = pd.DataFrame({
    'Metric': ['Row count', 'Null values (total)', 'Duplicate rows', 'PurchaseAmount dtype', 'JoinDate dtype', 'Gender categories'],
    'Before': [len(df), int(df.isnull().sum().sum()), int(df.duplicated().sum()), str(df['PurchaseAmount'].apply(type).mode()[0]), str(df['JoinDate'].dtype), df['Gender'].nunique(dropna=True)],
    'After': [len(df_clean), int(df_clean.isnull().sum().sum()), int(df_clean.duplicated().sum()), str(df_clean['PurchaseAmount'].dtype), str(df_clean['JoinDate'].dtype), df_clean['Gender'].nunique()]
})
summary

                 Metric         Before           After
0             Row count            625             601
1   Null values (total)            208               0
2        Duplicate rows             24               1
3  PurchaseAmount dtype  <class 'str'>         float64
4        JoinDate dtype            str  datetime64[us]
5     Gender categories              6               3

## 9. Save Cleaned Dataset

In [12]:
df_clean.to_csv('cleaned_customer_sales.csv', index=False)
print('Saved cleaned_customer_sales.csv —', df_clean.shape)
df_clean.head(10)

Saved cleaned_customer_sales.csv — (601, 9)


   CustomerID        Name  Age  Gender       City   JoinDate  PurchaseAmount  \
0        1001  Customer_0   20  Female     Mumbai 2022-01-01         1613.36   
1        1002  Customer_1   57    Male  Bengaluru 2022-02-01         1725.24   
2        1003  Customer_2   51  Female  Bengaluru 2022-01-04         4711.68   
3        1004  Customer_3   39  Female      Delhi 2022-05-01         4121.57   
4        1005  Customer_4   39  Female     Mumbai 2022-01-07         2129.43   
5        1006  Customer_5   62  Female     Mumbai 2022-01-08         2172.78   
6        1007  Customer_6   20    Male     Mumbai 2022-01-10         1465.72   
7        1008  Customer_7   42  Female  Bengaluru 2022-01-11          782.68   
8        1009  Customer_8   26    Male     Mumbai 2022-01-13         2394.81   
9        1010  Customer_9   21    Male    Kolkata 2022-01-14         1601.93   

  ProductCategory  Rating  
0      Home Decor       4  
1        Clothing       5  
2     Electronics       3  
3      

## Conclusion

Starting from a 625-row dataset with missing values, duplicate rows, mixed data types, inconsistent categorical formatting, mixed date formats, and outliers, we produced a clean, analysis-ready dataset by:

1. Quantifying every data quality issue before making any changes.
2. Imputing missing values with column-appropriate strategies (median for skewed numerics, mode for discrete ratings, an explicit `'Unknown'` category for missing location data — never a silently fabricated value).
3. Converting `PurchaseAmount` to a proper numeric dtype and `JoinDate` to a proper `datetime64` dtype.
4. Removing exact duplicate rows.
5. Standardising categorical text (case, whitespace) so that `'mumbai '`, `'MUMBAI'`, and `'Mumbai'` are no longer treated as three different cities.
6. Detecting and capping outliers in `Age` and `PurchaseAmount` using logical bounds and the IQR method, preserving row count while preventing distorted downstream statistics.

The cleaned dataset (`cleaned_customer_sales.csv`) is now ready for exploratory analysis, visualisation, or modelling.